# Exploración de datos de Instacart con Python y Pandas

**Actividad:** GA1-220501094-01-AA1-EV04 · Fundamentos en Pandas — *Explorando Instacart: descubre patrones de compra*
**Programa:** Análisis y Desarrollo de Software · SENA · Ficha 3223873
**Aprendiz:** _escribe aquí tu nombre completo_
**Fecha:** _escribe aquí la fecha de entrega_

## Momento 1 · Comprensión del problema y reconocimiento de los datos

### Descripción del problema
Instacart es una plataforma de compras de supermercado. Sus clientes generan información sobre **qué compran, cuándo compran y cada cuánto vuelven a comprar**. El equipo de análisis necesita saber qué contienen los archivos que se registran en la plataforma y cómo consultarlos para responder preguntas sobre el comportamiento de los clientes. Mi rol es el de analista de datos: hacer una **exploración inicial** (estructura, calidad, consultas puntuales e indicadores básicos) usando Python y Pandas.

**Pregunta orientadora:** ¿cómo utilizar Python y Pandas para explorar los datos de Instacart, identificar las características de los pedidos y productos, y obtener información relevante sobre los hábitos de compra de sus clientes?

### ¿Qué información almacena cada archivo?

| N.º | Archivo | Qué almacena | Columnas esperadas |
|---|---|---|---|
| 1 | `instacart_orders.csv` | Un registro por **pedido**: quién lo hizo, cuántos pedidos lleva ese usuario, qué día de la semana y a qué hora, y cuántos días pasaron desde su pedido anterior | `order_id`, `user_id`, `order_number`, `order_dow`, `order_hour_of_day`, `days_since_prior_order` |
| 2 | `products.csv` | Catálogo de **productos**, con el pasillo y el departamento al que pertenece cada uno | `product_id`, `product_name`, `aisle_id`, `department_id` |
| 3 | `aisles.csv` | Nombre de cada **pasillo** (tabla de referencia) | `aisle_id`, `aisle` |
| 4 | `departments.csv` | Nombre de cada **departamento** (tabla de referencia) | `department_id`, `department` |
| 5 | `order_products.csv` | Un registro por **artículo dentro de un pedido**: en qué posición entró al carrito y si es una recompra | `order_id`, `product_id`, `add_to_cart_order`, `reordered` |

### Columnas que identifican pedidos, usuarios y productos
- **Pedidos:** `order_id` (está en `orders` y en `order_products`).
- **Usuarios:** `user_id` (solo está en `orders`).
- **Productos:** `product_id` (está en `products` y en `order_products`).
- Para clasificar: `aisle_id` conecta `products` con `aisles`, y `department_id` conecta `products` con `departments`.

### Tres preguntas de análisis
1. **¿A qué hora del día se compra normalmente y cuántos pedidos se hacen a las 10 horas?** (se responde con `describe()` y filtrado lógico sobre `order_hour_of_day`).
2. **¿Cada cuántos días, en promedio, vuelven a comprar los clientes?** (se responde con `mean()` sobre `days_since_prior_order`).
3. **¿Qué parte de los artículos comprados corresponde a productos que el cliente ya había pedido antes?** (se responde con filtrado y conteo sobre `reordered`).

### ¿Por qué explorar los datos antes de analizarlos?
Antes de calcular cualquier indicador hay que saber **qué contiene cada archivo, cuántos registros tiene, de qué tipo es cada columna y dónde faltan datos**. Si se analiza sin explorar, se pueden cometer errores como interpretar el número de filas como número de clientes, promediar columnas con valores ausentes sin darse cuenta, o cruzar archivos con columnas que no corresponden. La exploración inicial da confianza en los resultados y deja claras las limitaciones.

## Momento 2 · Importación y exploración inicial de los archivos

In [47]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# Los archivos usan punto y coma (;) como separador, por eso sep=";".
# Se busca la carpeta con los CSV: ../data (si el cuaderno está en notebooks/), data/ o la carpeta actual.
RUTA = next(
    (c for c in [Path("../data"), Path("data"), Path(".")] if (c / "instacart_orders.csv").exists()),
    None,
)
assert RUTA is not None, "No encuentro los CSV: pon los 5 archivos en la carpeta data/ del proyecto."
print("Leyendo archivos desde:", RUTA.resolve())

fmt = lambda n: f"{n:,.0f}".replace(",", ".")   # 1234567 -> 1.234.567

Leyendo archivos desde: C:\Users\IK\Documents\instacart-analisis\data


In [48]:
orders = pd.read_csv(RUTA / "instacart_orders.csv", sep=";")
products = pd.read_csv(RUTA / "products.csv", sep=";")
aisles = pd.read_csv(RUTA / "aisles.csv", sep=";")
departments = pd.read_csv(RUTA / "departments.csv", sep=";")
order_products = pd.read_csv(RUTA / "order_products.csv", sep=";")

dataframes = {
    "instacart_orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}
print("Archivos cargados:", ", ".join(dataframes))

Archivos cargados: instacart_orders, products, aisles, departments, order_products


### Primeros y últimos cinco registros de cada DataFrame (`head()` y `tail()`)

In [49]:
for nombre, df in dataframes.items():
    print(f"\n===== {nombre} · primeros 5 registros =====")
    display(df.head())
    print(f"===== {nombre} · últimos 5 registros =====")
    display(df.tail())


===== instacart_orders · primeros 5 registros =====


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


===== instacart_orders · últimos 5 registros =====


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0



===== products · primeros 5 registros =====


...

,department_id,department
16,17,household
17,18,babies
18,19,snacks
19,20,deli
20,21,missing



===== order_products · primeros 5 registros =====


,order_id,product_id,add_to_cart_order,reordered
0,2141543,11440,17.0,0
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1


===== order_products · últimos 5 registros =====


,order_id,product_id,add_to_cart_order,reordered
4545002,577211,15290,12.0,1
4545003,1219554,21914,9.0,0
4545004,692640,47766,4.0,1
4545005,319435,691,8.0,1
4545006,1398151,28733,9.0,0


### Dimensiones de cada DataFrame (`shape`)

In [50]:
dimensiones = pd.DataFrame(
    {
        "DataFrame": list(dataframes),
        "filas": [df.shape[0] for df in dataframes.values()],
        "columnas": [df.shape[1] for df in dataframes.values()],
    }
)
display(dimensiones)

mas_registros = dimensiones.loc[dimensiones["filas"].idxmax(), "DataFrame"]
menos_columnas = dimensiones.loc[dimensiones["columnas"] == dimensiones["columnas"].min(), "DataFrame"].tolist()
print("DataFrame con más registros:", mas_registros, f"({fmt(dimensiones['filas'].max())} filas)")
print("DataFrame(s) con menos columnas:", menos_columnas, f"({dimensiones['columnas'].min()} columnas)")

,DataFrame,filas,columnas
0,instacart_orders,478967,6
1,products,49694,4
2,aisles,134,2
3,departments,21,2
4,order_products,4545007,4


DataFrame con más registros: order_products (4.545.007 filas)
DataFrame(s) con menos columnas: ['aisles', 'departments'] (2 columnas)


### Respuestas del Momento 2

- **¿Cuál contiene más registros?** `order_products`: es la tabla más grande porque guarda **un registro por cada artículo de cada pedido** (un pedido puede tener muchos artículos).
- **¿Cuál contiene menos columnas?** `aisles` y `departments`, con 2 columnas cada una (un identificador y un nombre). Son tablas de referencia.
- **Diferencia entre datos de pedidos y de productos:** `orders` describe el **comportamiento de compra** (quién, cuándo, cada cuánto: variables de tiempo y frecuencia). `products` describe el **catálogo** (qué se vende y cómo se clasifica). Los pedidos cambian cada día; el catálogo es relativamente estable.
- **¿Por qué importan las dimensiones?** Permiten saber de entrada cuántos datos hay, si el archivo se cargó completo (comparando con lo esperado), cuánta memoria puede necesitar el análisis y si una conclusión se apoya en pocos o en muchos registros.

## Momento 3 · Reconocimiento de la estructura y calidad inicial de los datos

In [51]:
for nombre, df in dataframes.items():
    print(f"\n===== {nombre} =====")
    print("Columnas:", df.columns.tolist())
    print("Forma (filas, columnas):", df.shape)
    print("\nTipos de dato (dtypes):")
    print(df.dtypes.to_string())
    print("\ninfo():")
    df.info()


===== instacart_orders =====
Columnas: ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
Forma (filas, columnas): (478967, 6)

Tipos de dato (dtypes):
order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64

info():
<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB

==

### Valores ausentes: registros vs. valores no nulos
`info()` reporta cuántos valores **no nulos** tiene cada columna. Si ese número es menor que el total de filas, faltan datos. Aquí se calcula esa diferencia para todas las columnas de los cinco archivos.

In [52]:
filas = []
for nombre, df in dataframes.items():
    for col in df.columns:
        filas.append(
            {
                "DataFrame": nombre,
                "columna": col,
                "dtype": str(df[col].dtype),
                "registros": len(df),
                "no_nulos": int(df[col].notna().sum()),
                "ausentes": int(df[col].isna().sum()),
            }
        )
calidad = pd.DataFrame(filas)
calidad["% ausente"] = (calidad["ausentes"] / calidad["registros"] * 100).round(2)

print("Columnas con valores ausentes:")
display(calidad.loc[calidad["ausentes"] > 0])

Columnas con valores ausentes:


,DataFrame,columna,dtype,registros,no_nulos,ausentes,% ausente
5,instacart_orders,days_since_prior_order,float64,478967,450148,28819,6.02
7,products,product_name,str,49694,48436,1258,2.53
16,order_products,add_to_cart_order,float64,4545007,4544171,836,0.02


### Tabla de diagnóstico

In [53]:
def observacion(nombre, df):
    notas = []
    ausentes = df.isna().sum()
    ausentes = ausentes[ausentes > 0]
    if ausentes.empty:
        notas.append("Sin valores ausentes")
    else:
        notas.append("Ausentes en " + ", ".join(f"{c} ({fmt(n)})" for c, n in ausentes.items()))
    dup = int(df.duplicated().sum())
    if dup:
        notas.append(f"{fmt(dup)} filas completamente duplicadas")
    return "; ".join(notas)

diagnostico = pd.DataFrame(
    {
        "archivo": [f"{n}.csv" for n in dataframes],
        "filas": [df.shape[0] for df in dataframes.values()],
        "columnas": [df.shape[1] for df in dataframes.values()],
        "observaciones": [observacion(n, df) for n, df in dataframes.items()],
    }
)
pd.set_option("display.max_colwidth", None)
display(diagnostico)

,archivo,filas,columnas,observaciones
0,instacart_orders.csv,478967,6,Ausentes en days_since_prior_order (28.819); 15 filas completamente duplicadas
1,products.csv,49694,4,Ausentes en product_name (1.258)
2,aisles.csv,134,2,Sin valores ausentes
3,departments.csv,21,2,Sin valores ausentes
4,order_products.csv,4545007,4,Ausentes en add_to_cart_order (836)


### Campos que relacionan los conjuntos de datos

| Campo | Une | Para qué sirve |
|---|---|---|
| `order_id` | `orders` ↔ `order_products` | Saber qué artículos tiene cada pedido |
| `product_id` | `products` ↔ `order_products` | Saber qué producto es cada artículo comprado |
| `aisle_id` | `products` ↔ `aisles` | Obtener el nombre del pasillo |
| `department_id` | `products` ↔ `departments` | Obtener el nombre del departamento |
| `user_id` | solo en `orders` | Identificar al cliente que hizo cada pedido |

### Respuestas del Momento 3
- **`int64`, `float64` y `object`:** `int64` guarda números enteros (identificadores, contadores, horas); `float64` guarda números decimales y además es el tipo que Pandas usa cuando una columna entera tiene valores ausentes (`NaN`); `object` (o `string`) guarda texto, como los nombres de productos, pasillos y departamentos. En esta versión de pandas (3.x) las columnas de texto aparecen como `str` en lugar de `object`; es el mismo tipo de dato (texto) con otro nombre, y en versiones anteriores se muestra como `object`.
- **¿Qué da `info()` que no da `shape`?** `shape` solo dice cuántas filas y columnas hay. `info()` además muestra el nombre y el tipo de cada columna, **cuántos valores no nulos tiene** (permite detectar ausentes) y el uso de memoria.
- **¿Dónde hay valores ausentes?** Hay ausentes en `days_since_prior_order` (28.819 registros, 6,02 %), `product_name` (1.258, 2,53 %) y `add_to_cart_order` (836, 0,02 %). Las demás columnas están completas.
- **Dificultades que pueden causar:** los promedios y conteos pueden calcularse sobre menos datos de los que se cree; los filtros con comparaciones (`>`, `==`) descartan en silencio las filas con `NaN`; un identificador ausente rompe los cruces entre tablas; y una columna entera con ausentes cambia a `float64`, lo que puede confundir.

*En esta fase no se corrige ni se elimina ningún registro: el objetivo es reconocer las características y limitaciones de los datos.*

## Momento 4 · Indexación y filtrado lógico de la información

Se usa `loc[]` para todas las consultas. La función auxiliar `resumen()` muestra cuántos registros devolvió el filtro y las primeras filas.

In [54]:
def resumen(resultado, original, n=5):
    total = len(original)
    print(f"Registros devueltos: {fmt(len(resultado))} de {fmt(total)} ({len(resultado) / total * 100:.2f} %)")
    display(resultado.head(n))

### Ejercicio 1 · Seleccionar únicamente `product_id` y `product_name` (products)

In [55]:
ej1 = products.loc[:, ["product_id", "product_name"]]
resumen(ej1, products)

Registros devueltos: 49.694 de 49.694 (100.00 %)


,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce
4,5,Green Chile Anytime Sauce


**Explicación:** es una **selección de columnas**: se conservan todas las filas, pero solo las dos columnas pedidas. La cantidad de registros no cambia.

### Ejercicio 2 · Productos con `department_id` igual a 1 (products)

In [56]:
ej2 = products.loc[products["department_id"] == 1]
resumen(ej2, products)

Registros devueltos: 4.008 de 49.694 (8.07 %)


,product_id,product_name,aisle_id,department_id
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
29,30,Three Cheese Ziti- Marinara with Meatballs,38,1


**Explicación:** es un **filtro de filas**: solo quedan los productos del departamento 1 (`frozen`, según `departments.csv`). Todas las columnas se mantienen.

### Ejercicio 3 · Pedidos realizados a las 10 horas (orders)

In [57]:
ej3 = orders.loc[orders["order_hour_of_day"] == 10]
resumen(ej3, orders)

Registros devueltos: 40.578 de 478.967 (8.47 %)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
28,133707,182261,1,3,10,NaN
36,3378084,127629,21,1,10,20.0
49,2161799,87345,7,0,10,24.0
51,1739741,118405,48,2,10,3.0


**Explicación:** se queda con los pedidos cuya hora de compra es exactamente 10. La salida indica cuántos pedidos se hacen en esa hora y qué porción del total representan.

### Ejercicio 4 · Pedidos con `order_dow` igual a 0 (orders)

In [58]:
ej4 = orders.loc[orders["order_dow"] == 0]
resumen(ej4, orders)

Registros devueltos: 84.090 de 478.967 (17.56 %)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
3,1768857,82516,56,0,20,10.0
5,1843871,20084,5,0,11,15.0
7,2521283,97711,2,0,18,22.0
23,3406282,19356,21,0,0,14.0
25,1895419,169793,2,0,11,8.0


**Explicación:** filtra los pedidos del día de la semana codificado como 0 (en este conjunto de datos, `order_dow` va de 0 a 6).

### Ejercicio 5 · Pedidos con `order_number` superior a 10 (orders)

In [59]:
ej5 = orders.loc[orders["order_number"] > 10]
resumen(ej5, orders)

Registros devueltos: 244.072 de 478.967 (50.96 %)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
3,1768857,82516,56,0,20,10.0
6,2511640,148233,40,3,8,2.0
8,2266494,34954,40,5,15,4.0
10,1162566,43738,33,4,11,2.0


**Explicación:** son los pedidos de usuarios que ya llevan más de 10 compras, es decir, clientes con historial de compra más largo.

### Ejercicio 6 · Pedidos con `days_since_prior_order` superior a 15 (orders)

In [60]:
ej6 = orders.loc[orders["days_since_prior_order"] > 15]
resumen(ej6, orders)

Registros devueltos: 110.706 de 478.967 (23.11 %)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
4,3007858,196724,2,4,12,17.0
7,2521283,97711,2,0,18,22.0
14,2843219,74699,4,6,11,30.0
26,1844247,122558,4,6,18,30.0


**Explicación:** son pedidos hechos más de 15 días después del anterior (clientes que compran con menos frecuencia). Las filas con `NaN` en esa columna **no entran** en el resultado, porque `NaN > 15` es falso.

### Ejercicio 7 · Registros con `reordered` igual a 1 (order_products)

In [61]:
ej7 = order_products.loc[order_products["reordered"] == 1]
resumen(ej7, order_products)

Registros devueltos: 2.683.838 de 4.545.007 (59.05 %)


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1
5,2456893,21616,4.0,1


**Explicación:** devuelve los artículos que son una **recompra**, es decir, productos que el cliente ya había pedido antes.

### Ejercicio 8 · Productos en la primera posición del carrito, `add_to_cart_order` igual a 1 (order_products)

In [62]:
ej8 = order_products.loc[order_products["add_to_cart_order"] == 1]
resumen(ej8, order_products)

Registros devueltos: 450.046 de 4.545.007 (9.90 %)


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
14,1961225,37553,1.0,1
16,639939,10017,1.0,1
23,750040,8518,1.0,0


**Explicación:** queda un artículo por pedido: el primero que el cliente agregó al carrito. Por eso la cantidad de registros se acerca a la cantidad de pedidos con artículos registrados.

### Filtros con condiciones simultáneas

In [63]:
# Condición A: pedidos a las 10 horas Y con order_number superior a 10
combinado_1 = orders.loc[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)]
print("Pedidos a las 10 horas y con order_number > 10")
resumen(combinado_1, orders)
print("Referencia: solo hora 10 ->", fmt(len(ej3)), "| solo order_number > 10 ->", fmt(len(ej5)))

Pedidos a las 10 horas y con order_number > 10
Registros devueltos: 21.537 de 478.967 (4.50 %)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
36,3378084,127629,21,1,10,20.0
51,1739741,118405,48,2,10,3.0
92,60724,110808,78,0,10,2.0
140,3156095,113920,14,2,10,2.0


Referencia: solo hora 10 -> 40.578 | solo order_number > 10 -> 244.072


In [64]:
# Condición B: productos del departamento 1 Y con aisle_id superior a 50
combinado_2 = products.loc[(products["department_id"] == 1) & (products["aisle_id"] > 50)]
print("Productos del departamento 1 con aisle_id > 50")
resumen(combinado_2, products)
print("Referencia: solo departamento 1 ->", fmt(len(ej2)))

Productos del departamento 1 con aisle_id > 50
Registros devueltos: 1.618 de 49.694 (3.26 %)


,product_id,product_name,aisle_id,department_id
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
45,46,Raisin Cinnamon Bagels 5 count,58,1
127,128,Organic Wild Blueberries,116,1


Referencia: solo departamento 1 -> 4.008


### Respuestas del Momento 4
- **Seleccionar columnas vs. filtrar filas:** seleccionar columnas reduce el **ancho** de la tabla (qué variables veo); filtrar filas reduce el **largo** (qué registros veo). Se pueden combinar: `df.loc[condición, ["col1", "col2"]]`.
- **Ventajas de `loc[]`:** permite indicar en una sola instrucción las filas (por condición lógica o etiqueta) y las columnas que se quieren; el código es explícito y legible, y es la forma recomendada para consultar (y luego modificar) datos sin ambigüedades.
- **Qué pasa con la cantidad de registros al aplicar varias condiciones:** con `&` (Y) el resultado **nunca es mayor** que el de cada condición por separado, porque un registro debe cumplirlas todas. Se ve en las líneas de «Referencia» de los dos filtros combinados.
- **Por qué importa usar bien los operadores lógicos:** en Pandas se usan `&` (y), `|` (o) y `~` (no), y **cada condición va entre paréntesis** por la precedencia de operadores. Usar `and`/`or` genera error, y confundir `&` con `|` cambia por completo el resultado (con `|` se amplía el conjunto en vez de reducirlo).

## Momento 5 · Obtención e interpretación de indicadores básicos

In [65]:
registros_pedidos = orders.shape[0]
pedidos_unicos = orders["order_id"].nunique()
usuarios_unicos = orders["user_id"].nunique()
registros_productos = products.shape[0]
prom_order_number = orders["order_number"].mean()
prom_dias = orders["days_since_prior_order"].mean()          # mean() ignora los NaN
reordered_1 = (order_products["reordered"] == 1).sum()
suma_reordered = order_products["reordered"].sum()
total_articulos = order_products.shape[0]

print("Registros en orders          :", fmt(registros_pedidos))
print("Pedidos distintos (order_id) :", fmt(pedidos_unicos))
print("Usuarios distintos (user_id) :", fmt(usuarios_unicos))
print("Registros en products        :", fmt(registros_productos))
print("Promedio de order_number     :", round(prom_order_number, 2))
print("Promedio days_since_prior    :", round(prom_dias, 2))
print("Registros con reordered == 1 :", fmt(reordered_1))
print("Suma de reordered            :", fmt(suma_reordered))

Registros en orders          : 478.967
Pedidos distintos (order_id) : 478.952
Usuarios distintos (user_id) : 157.437
Registros en products        : 49.694
Promedio de order_number     : 17.16
Promedio days_since_prior    : 11.1
Registros con reordered == 1 : 2.683.838
Suma de reordered            : 2.683.838


In [66]:
print("describe() de order_hour_of_day")
display(orders["order_hour_of_day"].describe().to_frame())
print("describe() de add_to_cart_order")
display(order_products["add_to_cart_order"].describe().to_frame())

describe() de order_hour_of_day


,order_hour_of_day
count,478967.000000
mean,13.446676
std,4.224986
min,0.000000
25%,10.000000
50%,13.000000
75%,16.000000
max,23.000000


describe() de add_to_cart_order


,add_to_cart_order
count,4.544171e+06
mean,8.351669e+00
std,7.080368e+00
min,1.000000e+00
25%,3.000000e+00
50%,6.000000e+00
75%,1.100000e+01
max,6.400000e+01


### Registros, pedidos y usuarios no son lo mismo
`orders` tiene una fila por pedido, pero **un mismo usuario hace muchos pedidos**, y un pedido puede aparecer repetido si el archivo tiene duplicados. Por eso el número de filas no es el número de clientes.

In [67]:
cantidades = pd.DataFrame(
    {
        "concepto": ["Registros (filas) en orders", "Pedidos distintos (order_id)", "Usuarios distintos (user_id)"],
        "cantidad": [registros_pedidos, pedidos_unicos, usuarios_unicos],
    }
)
cantidades["cantidad"] = cantidades["cantidad"].map(fmt)
display(cantidades)

print("Pedidos por usuario (promedio):", round(pedidos_unicos / usuarios_unicos, 2))
print("Filas duplicadas en orders    :", fmt(orders.duplicated().sum()))

,concepto,cantidad
0,Registros (filas) en orders,478.967
1,Pedidos distintos (order_id),478.952
2,Usuarios distintos (user_id),157.437


Pedidos por usuario (promedio): 3.04
Filas duplicadas en orders    : 15


### Tabla de indicadores: pregunta, valor e interpretación

In [68]:
hora = orders["order_hour_of_day"]
cart = order_products["add_to_cart_order"]

indicadores = pd.DataFrame(
    [
        (
            "¿Cuántos registros de pedidos contiene el archivo?",
            fmt(registros_pedidos),
            f"Hay {fmt(registros_pedidos)} filas en orders. Corresponden a {fmt(pedidos_unicos)} pedidos distintos y a {fmt(usuarios_unicos)} usuarios distintos: filas, pedidos y clientes son cantidades diferentes.",
        ),
        (
            "¿Cuántos registros de productos contiene el catálogo?",
            fmt(registros_productos),
            "Es el tamaño del catálogo. Cada fila es un producto distinto con su pasillo y departamento.",
        ),
        (
            "¿Cuál es el promedio de order_number?",
            f"{prom_order_number:.2f}",
            f"En promedio, un pedido es el n.º {prom_order_number:.1f} de su usuario: hay mucho historial de recompra por cliente.",
        ),
        (
            "¿Cuál es el promedio de days_since_prior_order (valores disponibles)?",
            f"{prom_dias:.2f}",
            f"Los clientes vuelven a comprar, en promedio, cada {prom_dias:.1f} días. Se calcula solo con los {fmt(orders['days_since_prior_order'].notna().sum())} pedidos que tienen dato.",
        ),
        (
            "¿Cuántos registros de artículos tienen reordered igual a 1?",
            fmt(reordered_1),
            f"{fmt(reordered_1)} de {fmt(total_articulos)} artículos ({reordered_1 / total_articulos * 100:.1f} %) son recompras.",
        ),
        (
            "¿Cuál es la suma de la columna reordered?",
            fmt(suma_reordered),
            "Coincide con el conteo anterior porque reordered solo toma los valores 0 y 1: sumar equivale a contar los unos.",
        ),
        (
            "¿Qué características estadísticas presenta order_hour_of_day?",
            f"media {hora.mean():.2f} · mediana {hora.median():.0f} · mín {hora.min()} · máx {hora.max()}",
            f"La hora típica de compra ronda las {hora.median():.0f} h; el 50 % central de los pedidos se hace entre las {hora.quantile(.25):.0f} h y las {hora.quantile(.75):.0f} h.",
        ),
        (
            "¿Qué características estadísticas presenta add_to_cart_order?",
            f"media {cart.mean():.2f} · mediana {cart.median():.0f} · mín {cart.min():.0f} · máx {cart.max():.0f}",
            f"Un artículo entra al carrito, en promedio, en la posición {cart.mean():.1f}; el pedido más largo llega a {cart.max():.0f} artículos. Se calcula con {fmt(cart.notna().sum())} valores disponibles.",
        ),
    ],
    columns=["Pregunta analizada", "Valor obtenido", "Interpretación"],
)
display(indicadores)

,Pregunta analizada,Valor obtenido,Interpretación
0,¿Cuántos registros de pedidos contiene el archivo?,478.967,"Hay 478.967 filas en orders. Corresponden a 478.952 pedidos distintos y a 157.437 usuarios distintos: filas, pedidos y clientes son cantidades diferentes."
1,¿Cuántos registros de productos contiene el catálogo?,49.694,Es el tamaño del catálogo. Cada fila es un producto distinto con su pasillo y departamento.
2,¿Cuál es el promedio de order_number?,17.16,"En promedio, un pedido es el n.º 17.2 de su usuario: hay mucho historial de recompra por cliente."
3,¿Cuál es el promedio de days_since_prior_order (valores disponibles)?,11.10,"Los clientes vuelven a comprar, en promedio, cada 11.1 días. Se calcula solo con los 450.148 pedidos que tienen dato."
4,¿Cuántos registros de artículos tienen reordered igual a 1?,2.683.838,2.683.838 de 4.545.007 artículos (59.1 %) son recompras.
5,¿Cuál es la suma de la columna reordered?,2.683.838,Coincide con el conteo anterior porque reordered solo toma los valores 0 y 1: sumar equivale a contar los unos.
6,¿Qué características estadísticas presenta order_hour_of_day?,media 13.45 · mediana 13 · mín 0 · máx 23,La hora típica de compra ronda las 13 h; el 50 % central de los pedidos se hace entre las 10 h y las 16 h.
7,¿Qué características estadísticas presenta add_to_cart_order?,media 8.35 · mediana 6 · mín 1 · máx 64,"Un artículo entra al carrito, en promedio, en la posición 8.4; el pedido más largo llega a 64 artículos. Se calcula con 4.544.171 valores disponibles."


## Momento 6 · Resolución del problema y síntesis individual

In [69]:
prop_reorden = reordered_1 / total_articulos * 100
pedidos_hora10 = len(ej3)
nulos_dias = int(orders["days_since_prior_order"].isna().sum())
nulos_nombre = int(products["product_name"].isna().sum())
nulos_cart = int(order_products["add_to_cart_order"].isna().sum())
dup_orders = int(orders.duplicated().sum())
dup_op = int(order_products.duplicated().sum())
primeros = int((orders["order_number"] == 1).sum())

texto = f"""
### 1. Preguntas de análisis y respuestas
1. **¿A qué hora se compra normalmente y cuántos pedidos hay a las 10 horas?** La hora mediana es las {hora.median():.0f} h (media {hora.mean():.1f} h) y a las 10 h se registran {fmt(pedidos_hora10)} pedidos ({pedidos_hora10 / registros_pedidos * 100:.1f} % del total).
2. **¿Cada cuántos días vuelven a comprar los clientes?** En promedio cada {prom_dias:.1f} días, calculado sobre los pedidos que tienen ese dato.
3. **¿Qué parte de los artículos son recompras?** {fmt(reordered_1)} de {fmt(total_articulos)} artículos, es decir {prop_reorden:.1f} %.

### 2. Principales hallazgos sobre pedidos y productos
- `orders` tiene {fmt(registros_pedidos)} filas, pero solo {fmt(pedidos_unicos)} pedidos distintos y {fmt(usuarios_unicos)} usuarios: cada usuario aparece en promedio en {pedidos_unicos / usuarios_unicos:.1f} pedidos del archivo (es una muestra de su historial, por eso `order_number` llega a valores mucho más altos).
- El catálogo tiene {fmt(registros_productos)} productos repartidos en {aisles.shape[0]} pasillos y {departments.shape[0]} departamentos.
- `order_products` es la tabla más grande ({fmt(total_articulos)} filas) porque guarda un registro por artículo comprado.
- Los pedidos se concentran en las horas centrales del día (mediana {hora.median():.0f} h, rango intercuartílico {hora.quantile(.25):.0f}–{hora.quantile(.75):.0f} h).

### 3. Cómo ayudaron la indexación y los filtros
Con `loc[]` se pudo pasar de tablas enormes a subconjuntos concretos: seleccionar solo las columnas necesarias (ejercicio 1), quedarse con los pedidos de una hora o de un día (ejercicios 3 y 4), con los clientes de más historial (ejercicio 5), con las recompras (ejercicio 7) o con el primer artículo de cada carrito (ejercicio 8). Al combinar condiciones con `&` se afinó la consulta y se comprobó que el resultado siempre es igual o menor que cada condición por separado.

### 4. Limitaciones identificadas en los datos
- `days_since_prior_order` tiene {fmt(nulos_dias)} valores ausentes. Hay {fmt(primeros)} pedidos con `order_number` = 1 (primer pedido del usuario, que no tiene pedido anterior), así que los ausentes {"coinciden con esos primeros pedidos" if nulos_dias == primeros else "están relacionados con los primeros pedidos, pero no coinciden exactamente con ellos"}. El promedio se calcula sin ellos.
- `products.product_name` tiene {fmt(nulos_nombre)} nombres ausentes: no se puede saber qué producto es.
- `add_to_cart_order` tiene {fmt(nulos_cart)} valores ausentes{", por lo que su estadística descriptiva usa menos registros que el total" if nulos_cart else ""}.
- Hay {fmt(dup_orders)} filas duplicadas en `orders` y {fmt(dup_op)} en `order_products`{": inflan los conteos si no se tratan" if (dup_orders or dup_op) else ""}.
- Los archivos no traen fechas calendario ni cantidades ni precios, solo día de la semana y hora, por lo que no se puede analizar estacionalidad ni ingresos.
- No se corrigió ningún dato: esta fase fue solo exploratoria.

### 5. Tres conclusiones sustentadas en los resultados
1. **Filas no equivalen a clientes:** {fmt(registros_pedidos)} filas en `orders` corresponden a {fmt(usuarios_unicos)} usuarios distintos, así que cualquier métrica «por cliente» debe usar `user_id` y `nunique()`.
2. **Los clientes son recurrentes:** el promedio de `order_number` es {prom_order_number:.1f}, vuelven a comprar cada {prom_dias:.1f} días y {prop_reorden:.1f} % de los artículos son recompras.
3. **Los datos sirven para explorar, pero necesitan limpieza antes de un análisis más profundo:** existen ausentes en tres archivos y duplicados en `orders` (sección 4) que habría que tratar antes de calcular indicadores finales.

### Respuesta a la pregunta orientadora
Python y Pandas permiten **cargar** los archivos (`read_csv` con `sep=";"`), **inspeccionarlos** (`head`, `tail`, `shape`, `dtypes`, `info`), **consultarlos** (`loc[]` con filtros lógicos) y **resumirlos** (`mean`, `sum`, `describe`, `nunique`). Con ello se identificaron las características de pedidos y productos y se obtuvieron indicadores de los hábitos de compra: hora típica de compra, frecuencia de recompra y proporción de artículos recomprados.
"""
display(Markdown(texto))


### 1. Preguntas de análisis y respuestas
1. **¿A qué hora se compra normalmente y cuántos pedidos hay a las 10 horas?** La hora mediana es las 13 h (media 13.4 h) y a las 10 h se registran 40.578 pedidos (8.5 % del total).
2. **¿Cada cuántos días vuelven a comprar los clientes?** En promedio cada 11.1 días, calculado sobre los pedidos que tienen ese dato.
3. **¿Qué parte de los artículos son recompras?** 2.683.838 de 4.545.007 artículos, es decir 59.1 %.

### 2. Principales hallazgos sobre pedidos y productos
- `orders` tiene 478.967 filas, pero solo 478.952 pedidos distintos y 157.437 usuarios: cada usuario aparece en promedio en 3.0 pedidos del archivo (es una muestra de su historial, por eso `order_number` llega a valores mucho más altos).
- El catálogo tiene 49.694 productos repartidos en 134 pasillos y 21 departamentos.
- `order_products` es la tabla más grande (4.545.007 filas) porque guarda un registro por artículo comprado.
- Los pedidos se concentran en las horas centrales del día (mediana 13 h, rango intercuartílico 10–16 h).

### 3. Cómo ayudaron la indexación y los filtros
Con `loc[]` se pudo pasar de tablas enormes a subconjuntos concretos: seleccionar solo las columnas necesarias (ejercicio 1), quedarse con los pedidos de una hora o de un día (ejercicios 3 y 4), con los clientes de más historial (ejercicio 5), con las recompras (ejercicio 7) o con el primer artículo de cada carrito (ejercicio 8). Al combinar condiciones con `&` se afinó la consulta y se comprobó que el resultado siempre es igual o menor que cada condición por separado.

### 4. Limitaciones identificadas en los datos
- `days_since_prior_order` tiene 28.819 valores ausentes. Hay 28.819 pedidos con `order_number` = 1 (primer pedido del usuario, que no tiene pedido anterior), así que los ausentes coinciden con esos primeros pedidos. El promedio se calcula sin ellos.
- `products.product_name` tiene 1.258 nombres ausentes: no se puede saber qué producto es.
- `add_to_cart_order` tiene 836 valores ausentes, por lo que su estadística descriptiva usa menos registros que el total.
- Hay 15 filas duplicadas en `orders` y 0 en `order_products`: inflan los conteos si no se tratan.
- Los archivos no traen fechas calendario ni cantidades ni precios, solo día de la semana y hora, por lo que no se puede analizar estacionalidad ni ingresos.
- No se corrigió ningún dato: esta fase fue solo exploratoria.

### 5. Tres conclusiones sustentadas en los resultados
1. **Filas no equivalen a clientes:** 478.967 filas en `orders` corresponden a 157.437 usuarios distintos, así que cualquier métrica «por cliente» debe usar `user_id` y `nunique()`.
2. **Los clientes son recurrentes:** el promedio de `order_number` es 17.2, vuelven a comprar cada 11.1 días y 59.1 % de los artículos son recompras.
3. **Los datos sirven para explorar, pero necesitan limpieza antes de un análisis más profundo:** existen ausentes en tres archivos y duplicados en `orders` (sección 4) que habría que tratar antes de calcular indicadores finales.

### Respuesta a la pregunta orientadora
Python y Pandas permiten **cargar** los archivos (`read_csv` con `sep=";"`), **inspeccionarlos** (`head`, `tail`, `shape`, `dtypes`, `info`), **consultarlos** (`loc[]` con filtros lógicos) y **resumirlos** (`mean`, `sum`, `describe`, `nunique`). Con ello se identificaron las características de pedidos y productos y se obtuvieron indicadores de los hábitos de compra: hora típica de compra, frecuencia de recompra y proporción de artículos recomprados.


### Verificación final
Antes de entregar: menú **Kernel → Restart Kernel and Run All Cells** (en VS Code: *Restart* y luego *Run All*). Todas las celdas deben ejecutarse sin errores y las respuestas escritas deben coincidir con las salidas.